# 06: TFPnP Paper Analysis

This notebook is a deep read of Wei et al. (2022) *"TFPnP: Tuning-free Plug-and-Play
Proximal Algorithm with Applications to Inverse Imaging Problems"*.

We do not just summarise the paper — we map every equation and algorithm step to
concrete Python code in `ct_tfpnp/`, reproduce Figure 1, and record every design
decision that shapes our implementation.

By the end you will understand:
- The MDP formulation: state, action, reward, transition
- Standard ADMM vs the inexact ADMM used in TFPnP and why it matters
- Algorithm 1 line by line, mapped to `TFPnPSolver.mini_batch_step()`
- The policy network architecture (Table 1) and how it maps to `ResNetActor_ADMM`
- The critic loss (eq. 15) and the two separate policy gradient updates
- The reward function (eq. 14) and the role of the η penalty
- Figure 1 reproduced with our implementation

This notebook runs on CSD3 with LION and GPU access.

## 1. Setup

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.ndimage import gaussian_filter
from copy import deepcopy

import ct_tfpnp
from LION.CTtools.ct_utils import make_operator
from LION.CTtools.ct_geometry import Geometry

device = torch.device("cuda")

print(f"ct_tfpnp : {ct_tfpnp.__version__}")
print(f"Device   : {device}")
print(f"GPU      : {torch.cuda.get_device_name()}")

# set consistent plotting style for all notebook cells
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "image.cmap": "gray",
    "image.interpolation": "nearest",
})

# create output directory for figures
output_dir = Path("../figures/06")
output_dir.mkdir(parents=True, exist_ok=True)

ct_tfpnp : 0.1.dev29+g44579dcd4.d20260608
Device   : cuda
GPU      : NVIDIA A100-SXM4-80GB


## 2. Paper Overview: The Core Idea in One Paragraph

PnP-ADMM produces high-quality reconstructions — but only if you choose the right
denoising strength $\sigma_k$ and penalty $\mu_k$ at every iteration, for every image.
These parameters are image-dependent and iteration-dependent, making manual tuning
impractical. Prior work uses heuristics (monotone schedules, handcrafted criteria) that
leave significant quality on the table.

**TFPnP's answer:** frame the parameter selection as a **Markov Decision Process** and
learn a policy $\pi$ that, given the current ADMM state, outputs the optimal $\sigma_k$,
$\mu_k$, and a termination decision — automatically, per image, per iteration.

The policy is trained by reinforcement learning. The reward at each step is the PSNR
increment produced by the chosen parameters, minus a small penalty $\eta$ for not stopping.
This encourages both quality improvement and computational efficiency.

**Two-stage training:**
1. Train the U-Net denoiser $\mathcal{H}_\sigma$ on image patches (supervised)
2. Freeze the denoiser; train the policy and critic networks end-to-end via RL

We are at stage 2. The denoiser is a pretrained module; our contribution is the policy.

## 3. The MDP Formulation (Section 3.1)

TFPnP frames PnP-ADMM parameter selection as a Markov Decision Process
$(\mathcal{S}, \mathcal{A}, p, r)$:

### State space $\mathcal{S}$

$$s_t = (x^k,\; z^k,\; u^k,\; \sigma_{\text{noise}},\; k/k_{\max}) \in \mathbb{R}^{5 \times H \times W}$$

The state is the full ADMM triple $(x, z, u)$ augmented with two auxiliary scalar
channels broadcast to spatial maps:
- $\sigma_{\text{noise}}$ — measurement noise level (tells the policy how noisy the input is)
- $k/k_{\max}$ — normalised iteration count (tells the policy how far along it is)

These auxiliary inputs are crucial (Table 9 of the paper shows they contribute ~0.2 dB).

### Action space $\mathcal{A}$

The action is decomposed: $a = (a_1, a_2)$

- $a_1 \in \{0, 1\}$ — **discrete** termination decision (0=continue, 1=stop)
- $a_2 = (\sigma_0, \ldots, \sigma_{m-1}, \mu_0, \ldots, \mu_{m-1})$ — **continuous** parameters
  for the next $m=5$ ADMM iterations

This two-part decomposition drives two separate sub-policies:
- $\pi_1$ — **stochastic** (samples $a_1$ from categorical), trained model-free
- $\pi_2$ — **deterministic** (outputs $a_2$ directly), trained model-based via backprop

### Transition function $p$

$$s_{t+1} = p(s_t, a_t)$$

Running $m=5$ ADMM iterations with the chosen $(\sigma, \mu)$ sequence:
$$x^{k+1} = \mathcal{H}_{\sigma_k}(z^k - u^k), \quad
  z^{k+1} = \text{Prox}_{\frac{1}{\mu_k}D}(x^{k+1} + u^k), \quad
  u^{k+1} = u^k + x^{k+1} - z^{k+1}$$

### Reward function $r$ (eq. 14)

$$r(s_t, a_t) = \underbrace{[\zeta(p(s_t, a_t)) - \zeta(s_t)]}_{\text{PSNR increment}} - \underbrace{\eta}_{\text{continuation penalty}}$$

where $\zeta(s)$ is the PSNR of $x$ in state $s$, and $\eta = 0.05$.

The $\eta$ term is key: if the PSNR gain from continuing is less than $\eta$, the policy
should prefer stopping. This is how TFPnP learns early stopping automatically.

In [4]:
# Implement the reward function (eq. 14) exactly as in the paper
def compute_reward(x_new, x_old, x_gt, eta=0.05, data_range=1.0):
    """
    r(s_t, a_t) = [PSNR(x_new, x_gt) - PSNR(x_old, x_gt)] - eta
    
    Args:
        x_new     : reconstruction after taking action (numpy array)
        x_old     : reconstruction before taking action
        x_gt      : ground truth image
        eta       : continuation penalty (paper uses 0.05)
        data_range: pixel value range (1.0 for normalised images)
    
    Returns:
        reward: scalar. Positive = improvement exceeded eta; negative = stop encouraged.
    """
    def psnr(a, b):
        mse = np.mean((a - b)**2)
        return 10 * np.log10(data_range**2 / mse) if mse > 0 else float('inf')
    
    psnr_new = psnr(x_new, x_gt)
    psnr_old = psnr(x_old, x_gt)
    return (psnr_new - psnr_old) - eta

# Demonstrate reward behaviour
print("Reward function behaviour (eta=0.05):")
print()
print("Scenario 1: large PSNR gain (+1.0 dB)")
print(f"  reward = {1.0 - 0.05:.2f}  → continue (positive reward)")
print()
print("Scenario 2: small PSNR gain (+0.03 dB, less than eta)")
print(f"  reward = {0.03 - 0.05:.2f}  → stop (negative reward)")
print()
print("Scenario 3: PSNR decreases (-0.2 dB)")
print(f"  reward = {-0.2 - 0.05:.2f}  → definitely stop (strongly negative)")
print()
print("This is how TFPnP learns to stop early: once gains drop below eta,")
print("the optimal action is termination (a1=1).")

Reward function behaviour (eta=0.05):

Scenario 1: large PSNR gain (+1.0 dB)
  reward = 0.95  → continue (positive reward)

Scenario 2: small PSNR gain (+0.03 dB, less than eta)
  reward = -0.02  → stop (negative reward)

Scenario 3: PSNR decreases (-0.2 dB)
  reward = -0.25  → definitely stop (strongly negative)

This is how TFPnP learns to stop early: once gains drop below eta,
the optimal action is termination (a1=1).


## 4. Standard ADMM vs Inexact ADMM

An important design decision in our implementation.

### Standard ADMM (exact z-step)

The z-step solves:
$$z^{k+1} = \arg\min_z \; \frac{1}{2}\|Az - y\|^2 + \frac{\mu}{2}\|z - (x+u)\|^2$$

For CT, the system $(A^\top A + \mu I)$ has no closed-form inverse. LION's standard solvers use **Conjugate Gradient (CG)** — accurate but not cleanly differentiable through the solver.

### Inexact ADMM (gradient descent z-step) — our implementation

TFPnP replaces the exact solve with gradient descent steps:
$$z^{k+1} = z^k - \alpha \nabla_z f(z^k), \quad f(z) = \frac{1}{2}\|Az - y\|^2 + \frac{\mu}{2}\|z - (x+u)\|^2$$

Our `ct_tfpnp/ct_ops/admm.py` implements this with normalised gradients:
```python
ATres = op.adjoint(op.forward(z) - y)
grad_data = ATres / (ATres.abs().max() + 1e-8)   # normalised direction
grad_prox = mu * (z - (x + u))
z = z - (grad_data + grad_prox)                   # 6 steps per z-step
```

**Why inexact?** The policy π₂ is trained **model-based**: gradients must flow from the reward back through the ADMM environment to the policy weights θ₂ (eq. 17). Gradient descent has clean, exact gradients for backpropagation. CG involves iterative solves whose differentiation is more complex.

The paper states this introduces negligible quality loss (Section 3.2). We accept this claim and use the inexact form throughout. Our notebook 05 results (25 dB peak with a Gaussian proxy denoiser) confirm the z-step is functioning correctly.

## 5. Figure 1 Reproduction — Per-Image σ Sensitivity

Figure 1 of the paper shows PSNR vs iterations for PnP-ADMM with different fixed σ values on multiple CT images. Each image peaks at a different σ value and at a different iteration count.

**We reproduced this figure in notebook 05, §11** using our real LION operator on LIDC-IDRI slices with the correct 30-view parallel beam geometry (724 detectors, unit spacing). The key qualitative findings match the paper:

1. Each image has a different optimal σ
2. Each image has a different optimal stopping iteration
3. High σ tends to peak early; low σ improves more slowly
4. Running to iteration 30 is suboptimal for most σ/image combinations

These four observations are exactly what TFPnP's policy learns to exploit. See `figures/05/10_per_image_sigma_sensitivity.pdf` for the reproduction.

## 6. Algorithm 1 — Line by Line, Mapped to Code

This is the most important cell in the notebook. We take every line of Algorithm 1
and show its exact implementation in `ct_tfpnp/training/solver.py`.

### Algorithm 1 (from the paper)

```
Input: Image dataset D, degradation operator g(·), learning rates l_θ, l_φ, weight β
1: Initialise network parameters θ, φ, φ̂ and state buffer B
2: for each training iteration do
3:     sample initial state s₀ from D via g(·)
4:     for environment step t ∈ [0, N) do
5:         aₜ ~ πθ(aₜ|sₜ)
6:         sₜ₊₁ ~ p(sₜ₊₁|sₜ, aₜ)
7:         B ← B ∪ {sₜ₊₁}
8:         break if the boolean outcome of aₜ equals to 1
9:     end for
10:    for each gradient step do
11:        sample states from the state buffer B
12:        θ₁ ← θ₁ + l_θ ∇θ₁ J(πθ)
13:        θ₂ ← θ₂ + l_θ ∇θ₂ J(πθ)
14:        φ ← φ − l_φ ∇φ L_φ
15:        φ̂ ← β φ + (1−β) φ̂
16:    end for
17: end for
Output: Learned policy network πθ
```

In [5]:
# Show the mapping between Algorithm 1 and TFPnPSolver.mini_batch_step()
# This is an annotated pseudo-implementation — not runnable, but shows correspondence

algorithm1_mapping = """
Algorithm 1                          →  ct_tfpnp/training/solver.py
─────────────────────────────────────────────────────────────────────────────────
Line 1: Initialise θ, φ, φ̂, B
  θ  = policy weights                →  self.model          (ResNetActor_ADMM)
  φ  = critic weights                →  self.critic         (ResNet_wobn)
  φ̂  = target critic weights         →  self.target_critic  (EMA copy of critic)
  B  = state buffer                  →  self.replay_buffer  (ReplayMemory)

Line 3: s₀ = g(x_gt)
  g(·) = forward project + FBP      →  x₀ = op.adjoint(y)  (FBP initialisation)
  s₀ = (x₀, z₀=x₀, u₀=0, σ_n, 0)  →  x,z,u initialised in mini_batch_step()

Lines 4-9: environment rollout
  for t in [0, N):                   →  for t in range(0, self.n_env_steps, self.m):
    aₜ ~ πθ(sₜ)                      →    stop_logits, sigma_seq, mu_seq = self.model(x,z,u,...)
    sₜ₊₁ = p(sₜ, aₜ)                →    x,z,u = self.admm_step(x,z,u,y,sigma,mu)  [×m times]
    B ← B ∪ {sₜ₊₁}                   →    self.replay_buffer.push(state, action, reward, next_state)
    break if a₁=1                    →    if stop.all(): break

Lines 10-16: gradient updates
  for each gradient step:            →  for _ in range(self.n_grad_steps):
    sample B                         →    batch = self.replay_buffer.sample(64)
    θ₁ ← θ₁ + l_θ ∇θ₁ J (eq. 16)   →    policy_loss_discrete.backward()   # REINFORCE
    θ₂ ← θ₂ + l_θ ∇θ₂ J (eq. 17)   →    policy_loss_continuous.backward()  # DDPG-style
    φ ← φ − l_φ ∇φ L_φ  (eq. 15)    →    critic_loss.backward()
    φ̂ ← β φ + (1−β) φ̂               →    EMA update of self.target_critic
─────────────────────────────────────────────────────────────────────────────────
"""
print(algorithm1_mapping)


Algorithm 1                          →  ct_tfpnp/training/solver.py
─────────────────────────────────────────────────────────────────────────────────
Line 1: Initialise θ, φ, φ̂, B
  θ  = policy weights                →  self.model          (ResNetActor_ADMM)
  φ  = critic weights                →  self.critic         (ResNet_wobn)
  φ̂  = target critic weights         →  self.target_critic  (EMA copy of critic)
  B  = state buffer                  →  self.replay_buffer  (ReplayMemory)

Line 3: s₀ = g(x_gt)
  g(·) = forward project + FBP      →  x₀ = op.adjoint(y)  (FBP initialisation)
  s₀ = (x₀, z₀=x₀, u₀=0, σ_n, 0)  →  x,z,u initialised in mini_batch_step()

Lines 4-9: environment rollout
  for t in [0, N):                   →  for t in range(0, self.n_env_steps, self.m):
    aₜ ~ πθ(sₜ)                      →    stop_logits, sigma_seq, mu_seq = self.model(x,z,u,...)
    sₜ₊₁ = p(sₜ, aₜ)                →    x,z,u = self.admm_step(x,z,u,y,sigma,mu)  [×m times]
    B ← B ∪ {sₜ₊₁}    

## 7. The Critic Loss (Equation 15)

The value network $V^\pi_\phi(s)$ is trained by minimising the TD error:

$$\mathcal{L}_\phi = \mathbb{E}_{s \sim B,\, a \sim \pi_\theta(s)}
\left[ \frac{1}{2} \left(
  r(s, a) + \gamma V^\pi_{\hat{\phi}}(p(s, a)) - V^\pi_\phi(s)
\right)^2 \right]$$

where:
- $B$ is the replay buffer (past ADMM states)
- $r(s, a)$ is the PSNR-increment reward (eq. 14)
- $\gamma = 0.99$ is the discount factor
- $\hat{\phi}$ is the **target critic** — an exponential moving average of $\phi$
  that stabilises training by providing a slowly-changing bootstrap target
- $p(s, a)$ is the next state after applying the ADMM step

The target value $r + \gamma V_{\hat{\phi}}(s')$ is computed with **no gradient**
(stop-grad on the target critic).

In [6]:
# Implement the critic loss (eq. 15) — this is what goes in _update_networks()

def critic_loss_fn(
    value_net,         # V^pi_phi: current critic
    target_value_net,  # V^pi_phi_hat: target critic (EMA, no grad)
    states,            # sampled from replay buffer
    rewards,           # r(s, a)
    next_states,       # p(s, a)
    gamma=0.99,
):
    """
    Critic loss from equation (15) of Wei et al.
    
    L_phi = E[ (1/2) * (r + gamma * V_hat(s') - V(s))^2 ]
    
    The target is computed with stop-gradient on target_value_net.
    """
    # Current value estimate V_phi(s)
    v_current = value_net(*states)            # (B, 1)
    
    # Bootstrap target: r + gamma * V_hat(s')  — no gradient
    with torch.no_grad():
        v_next = target_value_net(*next_states)   # (B, 1)
    
    td_target = rewards.unsqueeze(1) + gamma * v_next   # (B, 1)
    
    # TD error loss
    loss = 0.5 * F.mse_loss(v_current, td_target)
    return loss

print("critic_loss_fn() implements equation (15).")
print()
print("Key details:")
print("  - target_value_net uses stop-gradient (torch.no_grad())")
print("  - target_value_net weights φ̂ = β*φ + (1-β)*φ̂  (EMA, β=0.001)")
print("  - replay buffer B provides off-policy samples")
print("  - gamma=0.99 discounts future rewards")

critic_loss_fn() implements equation (15).

Key details:
  - target_value_net uses stop-gradient (torch.no_grad())
  - target_value_net weights φ̂ = β*φ + (1-β)*φ̂  (EMA, β=0.001)
  - replay buffer B provides off-policy samples
  - gamma=0.99 discounts future rewards


## 8. The Two Policy Gradient Updates (Equations 16 and 17)

The policy has two sub-policies with different gradient estimators:

### π₁: Discrete termination (REINFORCE, model-free) — eq. 16

$$\nabla_{\theta_1} J(\pi_\theta) = 
\mathbb{E}_{s \sim B,\, a \sim \pi_\theta(s)}
\left[ \nabla_{\theta_1} \log \pi_1(a_1|s) \cdot A^\pi(s, a) \right]$$

where $A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)$ is the **advantage function**.

Since $a_1$ is discrete (stop/continue), we cannot backprop through it directly.
REINFORCE provides an unbiased Monte Carlo gradient estimate using the log-probability
of the sampled action, weighted by the advantage.

### π₂: Continuous parameters (DDPG-style, model-based) — eq. 17

$$\nabla_{\theta_2} J(\pi_\theta) = 
\mathbb{E}_{s \sim B}\left[ \nabla_{\theta_2} V^\pi_\phi(p(s, \pi_2(s))) \right]$$

Since $a_2 = \pi_2(s)$ is continuous and the ADMM environment is differentiable,
we can backpropagate the value gradient directly through:
$$\theta_2 \leftarrow \theta_2 + l_\theta \nabla_{\theta_2} V_\phi(\text{ADMM}(s, \sigma, \mu))$$

This is the **model-based** part: the environment dynamics (ADMM step) is inside the
computational graph. This is why we implemented the z-step with gradient descent
rather than CG — the gradient must flow through it.

In [7]:
# Implement the two policy gradient losses

def policy_loss_discrete(
    policy_net,
    value_net,
    states,
    actions_a1,       # sampled discrete actions
    rewards,
    next_states,
    gamma=0.99,
):
    """
    REINFORCE gradient for π₁ (discrete termination) — equation (16).
    
    ∇θ₁ J = E[ ∇θ₁ log π₁(a₁|s) * A(s,a) ]
    
    where A(s,a) = r + γ V(s') - V(s)  (advantage)
    """
    stop_logits, _, _ = policy_net(*states)
    log_probs = F.log_softmax(stop_logits, dim=-1)              # (B, 2)
    log_prob_a1 = log_probs.gather(1, actions_a1.unsqueeze(1)).squeeze(1)  # (B,)
    
    # Advantage: A(s,a) = r + γV(s') - V(s)
    with torch.no_grad():
        v_s  = value_net(*states).squeeze(1)         # (B,)
        v_s_next = value_net(*next_states).squeeze(1) # (B,)
    advantage = rewards + gamma * v_s_next - v_s     # (B,)
    
    # REINFORCE loss: maximise E[log π₁(a₁|s) * A(s,a)]
    # → minimise -E[log π₁(a₁|s) * A(s,a)]
    loss = -(log_prob_a1 * advantage.detach()).mean()
    return loss


def policy_loss_continuous(
    policy_net,
    value_net,
    admm_step_fn,      # differentiable ADMM step
    states,
    sinograms,
):
    """
    DDPG-style gradient for π₂ (continuous σ, μ) — equation (17).
    
    ∇θ₂ J = E[ ∇θ₂ V_φ(p(s, π₂(s))) ]
    
    Gradients flow: θ₂ → (σ,μ) → ADMM step → next state → V_φ(next state)
    This requires the ADMM step to be differentiable w.r.t. σ and μ.
    """
    x, z, u, noise_level, iter_frac = states
    
    # Sample actions from π₂ (deterministic, differentiable)
    _, sigma_seq, mu_seq = policy_net(*states)   # (B, m), (B, m)
    
    # Run m ADMM steps with the policy's chosen parameters
    x_new, z_new, u_new = x.clone(), z.clone(), u.clone()
    for step_i in range(sigma_seq.shape[1]):
        sigma_i = sigma_seq[:, step_i]
        mu_i    = mu_seq[:, step_i]
        x_new, z_new, u_new = admm_step_fn(x_new, z_new, u_new, sinograms, sigma_i, mu_i)
    
    # Value of next state — gradient flows back through ADMM to θ₂
    n_iter = iter_frac + 1.0 / 6.0  # approximate next iter_frac
    next_state = (x_new, z_new, u_new, noise_level, n_iter.clamp(0, 1))
    v_next = value_net(*next_state)  # (B, 1)
    
    # Maximise V(s') → minimise -V(s')
    loss = -v_next.mean()
    return loss

print("Policy gradient losses implemented.")
print()
print("Key design points:")
print("  π₁ (discrete): REINFORCE — advantage-weighted log-prob")
print("  π₂ (continuous): DDPG-style — backprop through differentiable ADMM")
print("  The ADMM step must be differentiable w.r.t. σ and μ for π₂ to work")
print("  This is why z_step_gd() (gradient descent) is used, not z_step_cg()")

Policy gradient losses implemented.

Key design points:
  π₁ (discrete): REINFORCE — advantage-weighted log-prob
  π₂ (continuous): DDPG-style — backprop through differentiable ADMM
  The ADMM step must be differentiable w.r.t. σ and μ for π₂ to work
  This is why z_step_gd() (gradient descent) is used, not z_step_cg()


## 9. Network Architecture (Table 1)

The policy and value networks share the same ResNet-18 feature extractor backbone
(Table 1 of the paper), with different heads.

### Policy network (ResNetActor_ADMM)

```
Input: (x, z, u, noise_map, iter_map)   → (B, 5, H, W)
  │
  ├─ ResNet-18 backbone (modified first conv: 3→5 channels)
  │     conv1: 5→64, 7×7, stride 2
  │     layer1: 2× [64, 64] residual blocks, 32×32
  │     layer2: 2× [128, 128] residual blocks, 16×16
  │     layer3: 2× [256, 256] residual blocks, 8×8
  │     layer4: 2× [512, 512] residual blocks, 4×4
  │     avgpool: 4×4 → 1×1  →  512-dim feature vector
  │
  ├─ Termination head (π₁): Linear(512, 2) → softmax → P(stop), P(continue)
  └─ Parameter head (π₂):   Linear(512, 10) → sigmoid → (σ[5], μ[5]) scaled to ranges
```

### Value network (ResNet_wobn)

Same backbone structure but **without batch normalisation** (replaced by weight
normalisation + TReLU). Batch norm is unstable for value estimation because the
moving statistics shift as the policy changes during training.

Output: `Linear(512, 1)` → scalar $V^\pi(s)$

In [8]:
# Import and inspect our implementations
from ct_tfpnp.models.policy import ResNetActor_ADMM
from ct_tfpnp.models.critic import ResNet_wobn

policy = ResNetActor_ADMM(in_channels=5, n_action_steps=5)
critic = ResNet_wobn(in_channels=5)

# Count parameters
def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

p_total, p_train = count_params(policy)
c_total, c_train = count_params(critic)

print("Policy network (ResNetActor_ADMM):")
print(f"  Total parameters    : {p_total:,}")
print(f"  Trainable params    : {p_train:,}")
print()
print("Value network (ResNet_wobn):")
print(f"  Total parameters    : {c_total:,}")
print(f"  Trainable params    : {c_train:,}")
print()

# Test forward pass with correct shapes
B, H, W = 2, 64, 64
x   = torch.randn(B, 1, H, W)
z   = torch.randn(B, 1, H, W)
u   = torch.zeros(B, 1, H, W)
nse = torch.full((B,), 10.0)  # noise level
itr = torch.zeros(B)           # iteration fraction

stop_logits, sigma_seq, mu_seq = policy(x, z, u, nse, itr)
value = critic(x, z, u, nse, itr)

print("Forward pass shapes (B=2, H=W=64):")
print(f"  stop_logits : {stop_logits.shape}  ← 2-class softmax → P(stop), P(continue)")
print(f"  sigma_seq   : {sigma_seq.shape}    ← σ for m=5 ADMM steps, range [{sigma_seq.min():.1f}, {sigma_seq.max():.1f}]")
print(f"  mu_seq      : {mu_seq.shape}       ← μ for m=5 ADMM steps, range [{mu_seq.min():.3f}, {mu_seq.max():.3f}]")
print(f"  value       : {value.shape}         ← scalar V(s) per image")

Policy network (ResNetActor_ADMM):
  Total parameters    : 11,188,940
  Trainable params    : 11,188,940

Value network (ResNet_wobn):
  Total parameters    : 593,857
  Trainable params    : 593,857

Forward pass shapes (B=2, H=W=64):
  stop_logits : torch.Size([2, 2])  ← 2-class softmax → P(stop), P(continue)
  sigma_seq   : torch.Size([2, 5])    ← σ for m=5 ADMM steps, range [17.3, 35.8]
  mu_seq      : torch.Size([2, 5])       ← μ for m=5 ADMM steps, range [0.301, 0.542]
  value       : torch.Size([2, 1])         ← scalar V(s) per image


In [9]:
# Verify output ranges match paper specification
stop_probs = F.softmax(stop_logits, dim=-1)
print("Output range verification:")
print(f"  P(stop) range      : [{stop_probs[:,1].min():.3f}, {stop_probs[:,1].max():.3f}]  (should be in [0,1])")
print(f"  sigma range        : [{sigma_seq.min():.2f}, {sigma_seq.max():.2f}]  (paper: [1, 50])")
print(f"  mu range           : [{mu_seq.min():.4f}, {mu_seq.max():.4f}]  (paper: [0.01, 1.0])")
print()

# Verify gradient flow through policy (needed for π₂ model-based update)
x_grad = torch.randn(1, 1, 64, 64, requires_grad=True)
_, sigma_out, mu_out = policy(x_grad, torch.zeros_like(x_grad),
                               torch.zeros_like(x_grad),
                               torch.tensor([10.0]), torch.tensor([0.0]))
loss = sigma_out.sum() + mu_out.sum()
loss.backward()
print(f"Gradient flows through policy: {x_grad.grad is not None}")
print(f"  x.grad norm: {x_grad.grad.norm():.4f}  (non-zero = gradients flow correctly)")

Output range verification:
  P(stop) range      : [0.630, 0.648]  (should be in [0,1])
  sigma range        : [17.32, 35.80]  (paper: [1, 50])
  mu range           : [0.3006, 0.5416]  (paper: [0.01, 1.0])

Gradient flows through policy: True
  x.grad norm: 72.0227  (non-zero = gradients flow correctly)


## 10. State Representation: Why 5 Channels?

The input to both policy and value networks is a 5-channel tensor:

| Channel | Content | Why included |
|---------|---------|-------------|
| 0 | $x^k$ (current reconstruction) | Primary estimate — what we're trying to improve |
| 1 | $z^k$ (consensus variable) | Data-fidelity anchor — how far x has drifted from measurements |
| 2 | $u^k$ (dual variable) | Constraint violation — measures convergence quality |
| 3 | $\sigma_{\text{noise}} / \sigma_{\max}$ (broadcast) | Tells policy how noisy the input is |
| 4 | $k / k_{\max}$ (broadcast) | Tells policy how far along the optimisation is |

Table 9 of the paper shows that removing channels 3 and 4 degrades PSNR by ~0.2 dB. The policy uses them to be **problem-aware** — knowing the noise level and iteration count allows it to adapt its parameter choices accordingly.

A visualisation of all 5 channels from a real ADMM state is provided in **notebook 07, §X** (policy network development), where the state representation is tested end-to-end with the ResNet-18 policy.

## 11. The Full Training Pipeline: Putting It All Together

This section shows the complete data flow from raw CT image to policy update,
connecting every piece we have built.

In [10]:
# Print the complete training pipeline as an annotated diagram

pipeline = """
╔══════════════════════════════════════════════════════════════════════════╗
║              TFPnP Training Pipeline — One Iteration                     ║
╚══════════════════════════════════════════════════════════════════════════╝

LION DataLoader
  └─ yields (sinogram y, ground_truth x_gt)  — shape (B,1,30,724), (B,1,512,512)

INITIALISATION  [Algorithm 1, line 3]
  x₀ = op.adjoint(y)    ← FBP: backproject sinogram
  z₀ = x₀,  u₀ = 0

ENVIRONMENT ROLLOUT  [Algorithm 1, lines 4-9]
  for t = 0, 1, ..., N-1:  (N = 6 decision steps, giving 6 × m = 30 ADMM iterations)
  │
  ├─ Policy forward pass  [line 5: aₜ ~ πθ(aₜ|sₜ)]
  │     sₜ = (x, z, u, σ_noise, t/N)           ← 5-channel state
  │     (stop_logits, σ_seq, μ_seq) = policy(sₜ)
  │     a₁ ~ Categorical(softmax(stop_logits))  ← stochastic (model-free)
  │     (σ,μ) = (σ_seq, μ_seq)                  ← deterministic (model-based)
  │
  ├─ ADMM environment step × m  [line 6: sₜ₊₁ ~ p(sₜ₊₁|sₜ,aₜ)]
  │     for i in range(m=5):
  │         x = H_{σᵢ}(z - u)              ← x-step: denoiser prior
  │         z = (A^T A + μᵢ I)⁻¹(A^T y + μᵢ(x+u))  ← z-step: GD (inexact)
  │         u = u + x - z                  ← u-step: dual update
  │
  ├─ Reward  [eq. 14]
  │     r = [PSNR(x, x_gt) - PSNR(x_prev, x_gt)] - η
  │
  ├─ Store in replay buffer  [line 7: B ← B ∪ {sₜ₊₁}]
  │     replay_buffer.push(sₜ, (a₁, σ_seq, μ_seq), r, sₜ₊₁)
  │
  └─ Break if a₁ = 1  [line 8]

GRADIENT UPDATES  [Algorithm 1, lines 10-16]
  for _ in range(n_grad_steps=10):
  │
  ├─ Sample batch from replay_buffer  [line 11]
  │
  ├─ Critic loss (eq. 15)  [line 14]
  │     L_φ = ½ E[(r + γ V̂(s') - V(s))²]
  │     critic_optim.zero_grad(); L_φ.backward(); critic_optim.step()
  │
  ├─ Policy loss π₁ (eq. 16)  [line 12]
  │     L_θ₁ = -E[log π₁(a₁|s) * A(s,a)]
  │     policy_optim.zero_grad(); L_θ₁.backward(); policy_optim.step()
  │
  ├─ Policy loss π₂ (eq. 17)  [line 13]
  │     L_θ₂ = -E[V_φ(ADMM(s, π₂(s)))]
  │     policy_optim.zero_grad(); L_θ₂.backward(); policy_optim.step()
  │
  └─ EMA target critic update  [line 15]
        φ̂ ← β φ + (1-β) φ̂   (β = 0.001)
"""
print(pipeline)


╔══════════════════════════════════════════════════════════════════════════╗
║              TFPnP Training Pipeline — One Iteration                     ║
╚══════════════════════════════════════════════════════════════════════════╝

LION DataLoader
  └─ yields (sinogram y, ground_truth x_gt)  — shape (B,1,30,724), (B,1,512,512)

INITIALISATION  [Algorithm 1, line 3]
  x₀ = op.adjoint(y)    ← FBP: backproject sinogram
  z₀ = x₀,  u₀ = 0

ENVIRONMENT ROLLOUT  [Algorithm 1, lines 4-9]
  for t = 0, 1, ..., N-1:  (N = 6 decision steps, giving 6 × m = 30 ADMM iterations)
  │
  ├─ Policy forward pass  [line 5: aₜ ~ πθ(aₜ|sₜ)]
  │     sₜ = (x, z, u, σ_noise, t/N)           ← 5-channel state
  │     (stop_logits, σ_seq, μ_seq) = policy(sₜ)
  │     a₁ ~ Categorical(softmax(stop_logits))  ← stochastic (model-free)
  │     (σ,μ) = (σ_seq, μ_seq)                  ← deterministic (model-based)
  │
  ├─ ADMM environment step × m  [line 6: sₜ₊₁ ~ p(sₜ₊₁|sₜ,aₜ)]
  │     for i in range(m=5):
  │        

## 12. Hyperparameter Reference

Every hyperparameter from the paper, its value, its role, and where it is set in our code.

In [11]:
hyperparams = [
    # (symbol, value, description, location in code)
    ("m",      "5",      "ADMM steps per policy decision",            "TFPnPSolver.m"),
    ("N",      "6",      "Max decision steps (N×m = 30 ADMM iters)",  "TFPnPSolver.n_env_steps"),
    ("η",      "0.05",   "Continuation penalty in reward (eq. 14)",   "rewards.psnr_reward(scale=0.1)"),
    ("",       "",       "  (paper sweeps {0, 0.05, 0.1, 0.25} in Table 8)", ""),
    ("γ",      "0.99",   "Discount factor for future rewards",        "TFPnPSolver.gamma"),
    ("β",      "0.001",  "EMA weight for target critic update",       "TFPnPSolver.target_ema"),
    ("l_θ",    "1e-4",   "Policy learning rate (→5e-5 at iter 1600)", "TFPnPSolver.policy_optim (Adam)"),
    ("l_φ",    "5e-5",   "Critic learning rate (→1e-5 at iter 1600)", "TFPnPSolver.critic_optim (Adam)"),
    ("σ_min",  "1.0",    "Min denoising strength",                    "ResNetActor_ADMM.sigma_range"),
    ("σ_max",  "50.0",   "Max denoising strength",                    "ResNetActor_ADMM.sigma_range"),
    ("μ_min",  "0.01",   "Min penalty parameter",                     "ResNetActor_ADMM.mu_range"),
    ("μ_max",  "1.0",    "Max penalty parameter",                     "ResNetActor_ADMM.mu_range"),
    ("|B|",    "10,000", "Replay buffer capacity",                    "TFPnPSolver.replay_buffer"),
    ("n_grad", "10",     "Gradient updates per training iteration",    "TFPnPSolver.n_grad_steps"),
    ("batch",  "48",     "Training batch size (paper §4.1)",           "configs/tfpnp_default.yaml"),
    ("iters",  "2500",   "Training iterations (paper §4.1)",           "configs/tfpnp_default.yaml"),
]

print(f"{'Symbol':<10} {'Value':<12} {'Description':<50} {'Code location'}")
print("-" * 110)
for sym, val, desc, loc in hyperparams:
    print(f"{sym:<10} {val:<12} {desc:<50} {loc}")

print()
print("Note: N=6 decision steps × m=5 ADMM steps = 30 total ADMM iterations per episode.")
print("The paper (§4.1) uses 2500 training iterations with batch size 48.")
print("Learning rates are halved at iteration 1600 (§4.1).")

Symbol     Value        Description                                        Code location
--------------------------------------------------------------------------------------------------------------
m          5            ADMM steps per policy decision                     TFPnPSolver.m
N          6            Max decision steps (N×m = 30 ADMM iters)           TFPnPSolver.n_env_steps
η          0.05         Continuation penalty in reward (eq. 14)            rewards.psnr_reward(scale=0.1)
                          (paper sweeps {0, 0.05, 0.1, 0.25} in Table 8)   
γ          0.99         Discount factor for future rewards                 TFPnPSolver.gamma
β          0.001        EMA weight for target critic update                TFPnPSolver.target_ema
l_θ        1e-4         Policy learning rate (→5e-5 at iter 1600)          TFPnPSolver.policy_optim (Adam)
l_φ        5e-5         Critic learning rate (→1e-5 at iter 1600)          TFPnPSolver.critic_optim (Adam)
σ_min      1.0          M

## 13. Note on Inexact vs Exact ADMM Convergence

The paper claims the inexact (GD) z-step introduces negligible quality loss compared to exact (CG) solves (Section 3.2). We do not replicate this comparison empirically here because:

1. A fair comparison requires both solvers to use the same CT operator (`op.forward` / `op.adjoint` from LION), which is only available on CSD3
2. Our `ct_tfpnp/ct_ops/admm.py` implements the inexact form exclusively, as required for model-based RL (eq. 17)
3. Notebook 05 demonstrates that the inexact ADMM achieves a 10+ dB gain over FBP (14.4 → 25.0 dB), confirming the z-step functions correctly

If a direct comparison is needed for the report, it can be added as a CSD3 experiment using LION's CG solver as the baseline. For now, we accept the paper's claim and focus on the RL components.

## 14. Key Findings and Implementation Checklist

### Paper → Code mapping summary

| Paper concept | Equation | Our implementation | File |
|---------------|----------|-------------------|------|
| MDP state $s_t$ | §3.1 | 5-channel tensor | `solver.py` |
| Action $a_1$ (stop) | §3.1 | `stop_logits → softmax → Bernoulli` | `policy.py` |
| Action $a_2$ (σ, μ) | §3.1 | `param_head → sigmoid → scaled` | `policy.py` |
| Transition $p$ | §3.2 | `ADMMStep.forward()` | `ct_ops/admm.py` |
| Reward $r$ | eq. 14 | `psnr_reward()` | `training/rewards.py` |
| Critic loss $\mathcal{L}_\phi$ | eq. 15 | `critic_loss_fn()` | notebook → `solver.py` |
| Policy grad $\pi_1$ | eq. 16 | REINFORCE with advantage | notebook → `solver.py` |
| Policy grad $\pi_2$ | eq. 17 | backprop through ADMM | notebook → `solver.py` |
| Target critic update | line 15 | EMA, β=0.001 | `solver.py` |
| State buffer $B$ | §3.3 | `ReplayMemory` | `training/replay_buffer.py` |
| Policy network | Table 1 | `ResNetActor_ADMM` | `models/policy.py` |
| Value network | Table 1 | `ResNet_wobn` | `models/critic.py` |
| Denoiser $\mathcal{H}_\sigma$ | §3.2 | `UNetDenoiser2D` (pretrained) | `models/denoiser.py` |

### What needs to be completed in `solver.py`

The `_update_networks()` method in `TFPnPSolver` currently raises `NotImplementedError`.
After working through this notebook, you have all the pieces. Notebook 09 will implement
and test it interactively.

### What comes next

**Notebook 07 — Policy Network Development:** build the ResNet-18 policy from scratch,
verify shapes at every layer, load the pretrained denoiser, and run the first end-to-end
forward pass through the full ADMM environment.

In [12]:
# Final summary: print what each ct_tfpnp module implements from the paper
from ct_tfpnp.models.policy  import ResNetActor_ADMM
from ct_tfpnp.models.critic  import ResNet_wobn
from ct_tfpnp.models.denoiser import UNetDenoiser2D
from ct_tfpnp.ct_ops.admm    import ADMMStep
from ct_tfpnp.training.replay_buffer import ReplayMemory
from ct_tfpnp.training.rewards       import psnr_reward

print("ct_tfpnp module → paper concept:")
print()
print("  ResNetActor_ADMM   ← policy network πθ (Table 1 + §3.3)")
print("  ResNet_wobn        ← value network V^π_φ (Table 1 + §3.3)")
print("  UNetDenoiser2D     ← denoiser H_σ (§3.2, pretrained)")
print("  ADMMStep           ← transition function p (§3.2, differentiable)")
print("  ReplayMemory       ← state buffer B (§3.3)")
print("  psnr_reward        ← reward r(s,a) (eq. 14)")
print()
print("  TFPnPSolver        ← Algorithm 1 complete training loop")
print("    .mini_batch_step() ← lines 2-16")
print("    ._update_networks() ← lines 10-16  [to implement in nb 09]")

ct_tfpnp module → paper concept:

  ResNetActor_ADMM   ← policy network πθ (Table 1 + §3.3)
  ResNet_wobn        ← value network V^π_φ (Table 1 + §3.3)
  UNetDenoiser2D     ← denoiser H_σ (§3.2, pretrained)
  ADMMStep           ← transition function p (§3.2, differentiable)
  ReplayMemory       ← state buffer B (§3.3)
  psnr_reward        ← reward r(s,a) (eq. 14)

  TFPnPSolver        ← Algorithm 1 complete training loop
    .mini_batch_step() ← lines 2-16
    ._update_networks() ← lines 10-16  [to implement in nb 09]


notebook 06 is essentially annotated reading notes on the paper. It doesn't produce new results or run on real data. Here's what it actually contributes:

What it does well:

Maps every equation in the paper to a specific line of code in ct_tfpnp/ — this is genuinely useful when you're implementing solver.py and need to check "does my critic loss match eq. 15?"
The Algorithm 1 → code mapping (§6) and the paper→code table (§14) are reference material you'll come back to during implementation
The loss function definitions (§7–8) are basically pseudocode drafts for what goes into solver.py

What it doesn't do:

It doesn't run anything on real data
It doesn't produce figures for your report (notebook 05 already does the sensitivity analysis)
It doesn't train or test anything

My honest recommendation: keep it as a short reference notebook, but don't spend more time polishing it. Its value is as a bridge between "reading the paper" and "writing the training code." When you're implementing _update_networks() in solver.py, you'll open this notebook to check the loss functions and the Algorithm 1 mapping — that's its purpose.

If you want to make it more useful, the one thing worth adding would be a single code cell at the end that does a real end-to-end forward pass through the full pipeline on CSD3: load an image → forward project → FBP init → build state → policy forward pass → ADMM step → compute reward. 

That would verify everything actually connects before you start training. But that might belong better in notebook 07 or 08.

Shall we move on to the next notebook instead?